# CompVP Part 1 final submission
Hadis Nouri · Download CompVP_Part1_Hadis_Nouri_Final.zip from Output.
The retained 820 images and model-result files are checked against the completed Kaggle inference run. The ZIP includes the revised PowerPoint.

In [ ]:
from pathlib import Path
import importlib.util
import hashlib
import os
import shutil
import zipfile

working = Path('/kaggle/working')
inputs = Path('/kaggle/input')
payloads, expanded, originals = [], [], []
for directory, _, files in os.walk(inputs, followlinks=True):
    base = Path(directory)
    if 'revision_inputs.zip' in files:
        payloads.append(base / 'revision_inputs.zip')
    if 'package_revision.py' in files and 'presentation_verification.json' in files:
        expanded.append(base)
    if 'CompVP_Part1_Hadis_Nouri.zip' in files:
        originals.append(base / 'CompVP_Part1_Hadis_Nouri.zip')

reference_hash = '34b3bff221d3856d6602752aa0421cc9fd88bc9740aa2deb20524c90e6c19199'
verified = []
for candidate in sorted(set(originals), key=lambda path: len(path.parts)):
    with candidate.open('rb') as handle:
        if hashlib.file_digest(handle, 'sha256').hexdigest() == reference_hash:
            verified.append(candidate)
            break
assert verified, {'base_archives': [str(path) for path in originals]}
assert payloads or expanded, [str(path) for path in inputs.iterdir()]
revision = working / 'revision'
revision.mkdir(exist_ok=True)
if payloads:
    with zipfile.ZipFile(payloads[0]) as archive:
        assert archive.testzip() is None
        archive.extractall(revision)
else:
    shutil.copytree(expanded[0], revision, dirs_exist_ok=True)

spec = importlib.util.spec_from_file_location('package_revision', revision / 'package_revision.py')
generator = importlib.util.module_from_spec(spec)
spec.loader.exec_module(generator)
report = generator.package(
    verified[0], revision / generator.PRESENTATION,
    revision / 'submission_file_checksums.json', revision / 'manual_selection.json',
    working / generator.FINAL_ARCHIVE
)
for filename in [generator.PRESENTATION, 'CompVP_Part1_Hadis_Nouri.pdf', 'presentation_verification.json']:
    shutil.copyfile(revision / filename, working / filename)
print(report)
shutil.rmtree(revision)
